In [1]:
from continuousUtil import *
import sys
import time
import pandas as pd

sys.path.insert(0, "binding")
import p3gasus_continuous_cpp as p3cpp

def testTimeCpp(method, allPos):
    start = time.time()
    exGraph = method(allPos)
    end = time.time()
    edges = len(exGraph.edges()) - len(allPos[0]) * (len(exGraph.robot_list()) - 1)
    return edges, end - start


In [2]:
listOfMethods = [OriginalADG, SAGE, MAGE]
listOfCppMethods = [p3cpp.OriginalADG, p3cpp.SAGE, p3cpp.MAGE]


In [3]:
with open("Continuous Scenario Paths/10Agents_10fps", 'r') as f:
    try:
        data = json.load(f)
    except:
        print("Error loading JSON")
allPos = jsonToNpy(data, NUM_AGENTS=10)

In [4]:
rows = []

for method in listOfMethods:
    edges, seconds = testTime(method, allPos)
    rows.append({"Implementation": "Python", "Method": method.__name__, "Edges": edges, "Time (s)": seconds})

for method in listOfCppMethods:
    edges, seconds = testTimeCpp(method, allPos)
    rows.append({"Implementation": "C++", "Method": method.__name__, "Edges": edges, "Time (s)": seconds})

results = pd.DataFrame(rows).sort_values(["Method", "Implementation"]).reset_index(drop=True)
results["Time (s)"] = results["Time (s)"].round(6)
results


type: Time Taken - 51.58607339859009, Comms Length - 2360
type: Time Taken - 0.2727797031402588, Comms Length - 2360
type: Time Taken - 0.3050119876861572, Comms Length - 1254


In [5]:
exGraph = SAGE(allPos)
exGraph.fileWrite("Debug/")

exGraphCpp = p3cpp.SAGE(allPos)
exGraphCpp.file_write("Debug/")
